# Lessons 02-03: Generating ground truth

Groq free tier (8,000 tokens/min, 1,000 requests/day) is too small for all documents, so we generate questions for a **sample of 10 documents** with Groq and use the course's prepared `data/course/ground_truth-new.csv` (395 questions) for full-size evaluation, as the course suggests.

In [1]:
import os, json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

# Course: openai_client = OpenAI()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

In [2]:
from ingest import load_faq_data
documents = load_faq_data()

documents_llm = []
for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
len(documents)

161

In [3]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


## Structured output with Pydantic

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [5]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [6]:
user_prompt = json.dumps(doc)

messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

from evaluation_utils import MODEL

response = openai_client.responses.parse(
    model=MODEL,               # course: "gpt-5.4-mini"
    input=messages,
    text_format=Questions
)
result = response.output_parsed
result.questions

['Can I still enroll in the LLM Zoomcamp now that I just discovered it?',
 'If I join late, am I still eligible for a certificate?',
 'Do I need to submit a project to get the certificate after signing up late?',
 'Is there a deadline for project submissions to earn the certificate?',
 'What happens if I miss the current project submission window?']

## Reusable helper + cost

In [7]:
from evaluation_utils import llm_structured, calc_price

result, usage = llm_structured(openai_client, data_gen_instructions, user_prompt, Questions)
print(result.questions)
print(usage.input_tokens, usage.output_tokens)
calc_price(usage)

["Is it possible to enroll in the course now that I've just found out about it?", 'Can I still sign up for the LLM Zoomcamp if I discovered it recently?', 'Am I allowed to join the class after it has already started?', 'Do I need to meet any deadlines to get a certificate if I join late?', 'What are the requirements for getting the certificate if I enroll late?']
336 402


{'input_cost': 5.04e-05, 'output_cost': 0.0002412, 'total_cost': 0.0002916}

In [8]:
records = []
for q in result.questions:
    records.append({"question": q, "document": doc["id"]})
records

[{'question': "Is it possible to enroll in the course now that I've just found out about it?",
  'document': '74eb249bbf'},
 {'question': 'Can I still sign up for the LLM Zoomcamp if I discovered it recently?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to join the class after it has already started?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to meet any deadlines to get a certificate if I join late?',
  'document': '74eb249bbf'},
 {'question': 'What are the requirements for getting the certificate if I enroll late?',
  'document': '74eb249bbf'}]

## Batch (lesson 03): 10-document sample, 2 workers (free tier)

In [9]:
from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)
    out, usage = llm_structured_retry(openai_client, data_gen_instructions, user_prompt, Questions)
    results = []
    for q in out.questions:
        results.append({"question": q, "document": doc["id"]})
    return results, usage

In [10]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress, calc_total_price
import time

sample_docs = documents[:10]
t0 = time.time()
with ThreadPoolExecutor(max_workers=2) as pool:   # course: 6 workers
    results = map_progress(pool, sample_docs, generate_ground_truth)
print(f"{time.time() - t0:.0f} s")

ground_truth = []
usages = []
for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

print(len(ground_truth), "questions, cost $", round(calc_total_price(usages), 6))

  0%|          | 0/10 [00:00<?, ?it/s]

11 s
50 questions, cost $ 0.003308


In [11]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)
os.makedirs("data", exist_ok=True)
df_ground_truth.to_csv("data/ground_truth-groq-sample.csv", index=False)
df_ground_truth.head(10)

,question,document
0,Is it possible to enroll in the LLM Zoomcamp n...,74eb249bbf
1,Can I still sign up for the course and get the...,74eb249bbf
2,Do late joiners need to submit a project to ea...,74eb249bbf
3,Are there any deadlines for submitting the fin...,74eb249bbf
4,What happens if I enroll after the course star...,74eb249bbf
5,Do I need to wait for a confirmation email bef...,977bf7786c
6,"Is there a deadline to register, or can I just...",977bf7786c
7,Will my registration be checked against a list...,977bf7786c
8,What’s the purpose of the registration form if...,977bf7786c
9,Can I start the course and do assignments whil...,977bf7786c


## Course full file (395 questions, generated 2026-05-29 with gpt-5.4-mini)

In [12]:
df_course = pd.read_csv("data/course/ground_truth-new.csv")
print(len(df_course), "questions,", df_course["document"].nunique(), "documents")
missing = set(df_course["document"]) - {d["id"] for d in documents}
print("course document ids no longer in today's FAQ:", len(missing))
df_course.head()

395 questions, 78 documents
course document ids no longer in today's FAQ: 16


,question,document
0,Is it okay to join the course late if I just f...,74eb249bbf
1,Can I still take this course even if I missed ...,74eb249bbf
2,If I join after the course has already started...,74eb249bbf
3,Do I need to submit my project before submissi...,74eb249bbf
4,I’m a bit late to the course—what do I need to...,74eb249bbf
